# Part 5: Design Decisions and Analysis

This notebook provides detailed analysis for all 7 required sections, based on actual experimental results and visual inspection of generated outputs. 

## 1. Tokenization Strategy

We trained a Byte Pair Encoding (BPE) tokenizer using the HuggingFace tokenizers library with ByteLevel pre-tokenization and a vocabulary size of 4096 tokens (including 4 special tokens: PAD, BOS, EOS, UNK).

**Why BPE with ByteLevel:**

SVG code is XML-based text with highly repetitive substrings. Tags like `<path d="M`, attribute patterns like `fill="none" stroke="black"`, and coordinate sequences like `0.0 0.0 24.0 24.0` appear thousands of times across the corpus. BPE naturally learns to merge these frequent byte sequences into single tokens, compressing the input significantly. ByteLevel pre-tokenization ensures that any byte sequence can be represented without unknown token issues, which is important for SVG because it contains special characters, numeric values with decimals, and hexadecimal color codes that a word-level tokenizer would struggle with.

**Why vocabulary size 4096:**

SVG has a much smaller effective vocabulary than natural language. The entire domain consists of approximately 30 distinct tag names (svg, path, circle, rect, g, defs, etc.), around 50 common attribute names (d, fill, stroke, viewBox, transform, etc.), and numeric coordinate values. A vocabulary of 4096 is large enough to capture common multi-character patterns (such as encoding `<path d="M` as one or two tokens instead of 10+ bytes) while keeping the embedding table small enough that it does not dominate parameter counts for our smallest models.

For reference, the Tiny model has 1.35M total parameters. With vocab size 4096 and d_model=128, the embedding matrix alone is 4096 x 128 = 524,288 parameters, already 39% of the model. A larger vocabulary (say 16K or 32K as used in natural language) would make the embedding table disproportionately large relative to the transformer layers, distorting the scaling analysis since parameter increases would go to the embedding rather than the transformer blocks.

**Effect on sequence length and model performance:**

After tokenization, the sequence length distribution (see histogram) shows a roughly bell-shaped curve centered around 400-500 tokens:

| Statistic | Value |
|-----------|-------|
| Mean | 500 tokens |
| Median | 475 tokens |
| Std dev | 199 tokens |
| P25 | 354 tokens |
| P75 | 622 tokens |
| P95 | 889 tokens |
| Min | 93 tokens |
| Max | 1024 tokens (filter cutoff) |

We filtered out SVGs exceeding 1024 tokens, removing 48,019 files (17% of the 284,548 cleaned SVGs). This tradeoff eliminates the most complex SVGs but ensures all remaining sequences fit within a reasonable context window. The removed SVGs were typically highly detailed illustrations with many path commands.

**Preprocessing impact:**

Coordinate rounding to 1 decimal place was critical for tokenization quality. Without rounding, coordinates like 12.345678 and 12.345679 would produce completely different token sequences despite being visually identical. Rounding reduces the number of unique numeric substrings the tokenizer must learn, allowing it to allocate vocabulary capacity to more meaningful structural patterns.

The cleaning step (removing comments, metadata, collapsing whitespace) passed all 284,548 raw SVGs without removing any, indicating the source datasets were already well-formatted. The token length filter was the only reduction step, bringing the corpus from 284,548 to 236,529 files, yielding 115.9M training tokens, 1.18M validation tokens, and 1.19M test tokens.


## 2. Architecture Choices

We implemented a decoder-only transformer following the GPT-2/nanoGPT architecture with the following design choices:

- **Pre-LayerNorm**: LayerNorm is applied before attention and MLP blocks rather than after. Pre-norm improves training stability, particularly at higher learning rates, and is standard in modern architectures including GPT-2 and LLaMA.
- **GELU activation**: Used in the feed-forward network. GELU provides smoother gradients near zero and has become the default for transformer LMs since GPT-2.
- **Weight tying**: The token embedding and output projection share weights under standard parameterization, reducing parameter count. Under muP, weight tying is disabled because MuReadout applies its own width-dependent scaling.
- **Learned positional embeddings**: SVG has position-dependent patterns (xmlns at position 0, viewBox early) that learned embeddings can capture.

**Model sizes:**

| Name | Params | d_model | layers | heads | d_ff | Head dim |
|------|--------|---------|--------|-------|------|----------|
| Tiny | 1,350,400 | 128 | 4 | 4 | 512 | 32 |
| Small | 3,505,152 | 192 | 6 | 6 | 768 | 32 |
| Medium | 12,318,720 | 384 | 6 | 6 | 1536 | 64 |
| Large | 33,753,088 | 512 | 10 | 8 | 2048 | 64 |
| XL | 88,398,336 | 768 | 12 | 12 | 3072 | 64 |

The models span approximately 65x in parameter count. The d_ff is consistently 4x d_model. We varied both width and depth jointly. Small and Medium share depth (6 layers) but differ in width, while Large and XL have more layers (10 and 12).

**Context window:**

We used a context window of 256 tokens, shorter than the median SVG length of 475 tokens. This was a practical compromise: the XL model already uses 17.7 GB GPU memory at block_size=256, and doubling to 512 would roughly quadruple attention memory. The data loader samples random 256-token chunks from the concatenated stream, so the model sees all parts of longer SVGs across different training steps. The main limitation is that the model cannot attend across an entire complex SVG in a single forward pass, which limits learning of global structural relationships.


## 3. Training Decisions

**Optimizer:** AdamW with betas (0.9, 0.95) and weight decay 0.1. The beta2 of 0.95 provides more aggressive second-moment adaptation than the default 0.999, which helps with non-stationary transformer training dynamics.

**Batch size:** 64 sequences x 256 tokens = 16,384 tokens per gradient step, giving 7,103 steps per epoch over the 115.9M token training set.

**Learning rate sweep:**

| LR | Val Loss | Train Loss |
|----|----------|------------|
| 1e-05 | 3.0585 | 3.1112 |
| 3e-05 | 2.2450 | 2.2289 |
| 1e-04 | 1.8561 | 1.9226 |
| 3e-04 | 1.5252 | 1.5929 |
| 1e-03 | 1.3272 | 1.3587 |
| 3e-03 | 1.2550 | 1.2679 |
| **1e-02** | **1.2212** | **1.2830** |

The best LR was 0.01, the highest tested. The curve was still declining at 0.01, suggesting even higher LRs might work for the Tiny model. This is notably higher than typical NLP transformer LRs (1e-4 to 6e-4). SVG may tolerate higher LRs because the data has more regular structure and lower entropy than natural language. However, as discovered in the scaling study, this LR was catastrophically wrong for larger models under standard parameterization.

**Learning rate schedule:** Cosine decay with linear warmup over 5% of total steps (approximately 355 steps), decaying to 10% of peak LR.

**Regularization:** Dropout of 0.1 at attention weights, residual connections, and embedding layer. Gradient norm clipping at 1.0.

**Training duration:** 1 epoch (7,103 steps) per model for the scaling comparison. 5 epochs (35,515 steps, 6.3 hours) for the best model (XL with muP). The training curve shows smooth convergence from loss 5.0 down to 0.92, with train and validation loss tracking closely (no overfitting), indicating the model could benefit from additional epochs.


## 4. Scaling Insights

**Standard parameterization results (1 epoch, LR=0.01 fixed):**

| Model | Params | Val Loss | Tok/s | GPU MB |
|-------|--------|----------|-------|--------|
| Tiny | 1,350,400 | 1.2294 | 580,939 | 2,286 |
| Small | 3,505,152 | 1.0506 | 282,007 | 3,773 |
| Medium | 12,318,720 | **0.9470** | 142,500 | 5,152 |
| Large | 33,753,088 | 1.0668 | 59,643 | 10,136 |
| XL | 88,398,336 | 1.3667 | 25,906 | 17,687 |

**Non-monotonic scaling:** The most striking result is that the scaling curve is non-monotonic. Loss decreases from Tiny (1.23) to Medium (0.95), then increases to Large (1.07) and sharply to XL (1.37). The XL model, with 65x more parameters than Tiny, achieves WORSE loss than Tiny.

The SP training curves plot reveals why: the XL curve shows severe instability in early training, spiking to loss 4.3 around step 500 and oscillating violently for the first 2,500 steps before slowly descending. By step 7,103 it has reached 1.37 but is clearly still declining and far from convergence. The Large model also shows mild bumps. In contrast, Tiny, Small, and Medium converge smoothly within the first 2,000 steps.

This is direct evidence that LR=0.01 exceeds the stability threshold for models wider than d_model=384. The gradient update magnitude scales with width, so the same LR produces increasingly large effective steps for wider models.

**Power law fit:** L = 1,000,000 * N^(-1.162) + 1.110, R-squared = 0.041. The R-squared of 0.041 indicates essentially no fit. The extreme parameters are artifacts of trying to fit a non-monotonic curve with a monotonic function.

**Comparison with known scaling laws:** Kaplan et al. (2020) found smooth power-law scaling with alpha approximately 0.076 for models from 768 to 1.5B parameters. Our SP results demonstrate that without proper per-size LR tuning, scaling laws simply do not hold. This is itself an important finding: naive application of a fixed learning rate across model sizes can make it appear that larger models are worse, when in fact the hyperparameter is wrong.


## 5. Learning Rate Scaling Insights (SP vs muP)

**LR sweep comparison (Tiny model):** Both SP and muP select LR=0.01 as optimal with nearly identical validation loss (1.2212 vs 1.2246). The LR sweep comparison plot shows the two curves overlapping almost exactly. This is expected: muP matches SP at the base width.

**Scaling comparison (all models, 1 epoch, LR=0.01):**

| Model | SP Val Loss | muP Val Loss | Difference | Winner |
|-------|-------------|--------------|------------|--------|
| Tiny | 1.2294 | 1.2215 | +0.008 | muP (tiny edge) |
| Small | 1.0506 | 1.0738 | -0.023 | SP |
| Medium | 0.9470 | 1.0252 | -0.078 | SP |
| Large | 1.0668 | 0.9412 | +0.126 | muP |
| XL | 1.3667 | 0.9745 | +0.392 | muP (dramatic) |

**The crossover point** occurs between Medium and Large. Below Medium, SP actually performs better (the fixed LR happens to be near-optimal for those widths). Above Medium, muP dominates increasingly. At XL, the improvement is 0.392 in validation loss.

The muP training curves confirm this visually: all five model sizes converge smoothly with proper ordering (larger models achieve lower loss). There are no instability spikes, in stark contrast to the SP training curves where XL oscillates wildly.

**Power law comparison:**

| Method | Alpha | R-squared |
|--------|-------|-----------|
| SP | 1.162 | 0.041 (no fit) |
| muP | 0.866 | 0.960 (excellent) |

The muP scaling curve follows a clean power law. The exponent of 0.866 is much larger than the 0.076 from Kaplan et al., likely because our models are much smaller (1M-91M vs 768-1.5B), we train for only 1 epoch, and SVG is a more structured domain where additional parameters provide proportionally more benefit.

Note the muP XL (0.9745) shows a slight uptick from muP Large (0.9412), suggesting either 1 epoch is insufficient for XL to fully converge, or the scaling law is beginning to flatten.

**Extrapolation (using muP fit):** For a model with 10x more parameters than XL (884M params), the predicted validation loss is 0.9535 with 95% CI [0.888, 1.019]. The predicted improvement of only 0.021 from 91.5M to 884M suggests strong diminishing returns. The wide confidence interval reflects the uncertainty of extrapolating 10x beyond training data.

**Why muP works:** Under SP, gradient update magnitude for a layer scales with its width. LR=0.01 tuned for Tiny (d_model=128) produces effective updates for XL (d_model=768) that are roughly 6x larger than intended, causing the loss spikes visible in the training curves. muP corrects this through: (1) attention scaling 1/d instead of 1/sqrt(d), (2) MuReadout scaling the output layer inversely with width, and (3) MuAdamW applying per-layer LR multipliers to maintain constant update-to-weight ratios.


## 6. SVG-Specific Patterns

**Best model performance:** XL with muP (91.5M params, 5 epochs) achieved test perplexity 2.51, validation loss 0.9203, XML validity 82.6%, render rate 82.6%, and structural validity 82.6%. A perplexity of 2.51 is extremely low compared to natural language (GPT-2 achieves 20-30), reflecting the highly predictable structure of SVG code.

**Temperature effects (from visual inspection of generated samples):**

**T=0.5** (1/6 renderable): The model exhibits bimodal behavior, producing either extremely short sequences (26 tokens, immediate EOS) or maximum-length sequences (793 tokens, hitting the limit). The one renderable sample shows overlapping circles and curved lines. Most samples fail to render because they hit the token limit and get truncated mid-tag. T=0.5 concentrates probability mass too heavily on either EOS or the most common continuation pattern, with no middle ground.

**T=0.8** (2/6 renderable): Produces the most visually coherent, icon-like outputs. The two renderable samples show recognizable geometric shapes: one resembles a cup or funnel with rectangular and trapezoidal elements, another shows an L-shaped structure with vertical lines resembling a building. These have clean lines, centered compositions, and plausible object shapes. However, the low render rate (33%) means most T=0.8 samples still get truncated.

**T=1.0** (5/6 renderable, highest success rate): Shows high diversity but less structure. Outputs include overlapping organic curves, elongated swooping lines resembling calligraphy, a tall narrow rectangle, and circular bubble clusters. More diverse and more likely to render (varied token lengths avoid truncation), but the outputs look more like abstract art than recognizable icons. The shorter average token counts suggest T=1.0 produces more natural EOS stopping points.

**Temperature comparison (best sample per temperature):**
- T=0.5: Abstract tangle of overlapping circles and curves
- T=0.8: Clean geometric cup/funnel shape (most icon-like)
- T=1.0: Dense overlapping curves filling a circular region

The key finding: T=0.8 produces the most coherent outputs but with the lowest render success rate. T=1.0 has the highest render rate but produces more abstract outputs. This tension between visual coherence and renderability is important for practical SVG generation.

**Prefix completion analysis (from visual inspection):**

1. **Partial face** (circle + one eye, +569 tokens): The model kept the circle and eye dot, and added a curved arc below the eye resembling a mouth. It produced a simple face (circle, one eye, curved mouth), but did NOT add a second eye. The model learned that faces have curved features but did not fully capture bilateral symmetry.

2. **Open triangle** (+426 tokens): The model preserved the inverted V shape and added minimal additional line segments. The completion appears nearly identical to the prefix with slightly thicker strokes, suggesting the model closed the path conservatively.

3. **Group with rectangle** (+1 token): The model immediately closed the group tag with just 1 additional token, producing the rectangle with a slightly thicker border. This is the most minimal possible completion. The model learned that a single rectangle in a group is a valid icon, but missed the opportunity to add complexity.

4. **Star start** (+768 tokens, FAILED): Hit the maximum token limit without terminating. The model could not complete the polygon coordinate sequence correctly, getting stuck generating coordinates without ever closing the polygon tag. Polygon points require precise numeric pairs within a specific geometric pattern, which is the hardest generation task.

5. **Two circles** (+374 tokens): The best completion. The model kept both circles and added a third, smaller circle between and slightly below the originals. The result is a three-circle pattern resembling a molecular diagram. The model recognized the spatial pattern (two circles at the same height) and added a contextually appropriate element maintaining the theme.

**Scale differences:** The training losses suggest Tiny/Small learn basic SVG syntax, Medium captures most patterns at this data scale (with the largest loss improvement from Small to Medium), and Large/XL with muP provide marginal additional refinement. The diminishing returns above 30M parameters suggest 115.9M training tokens may be insufficient to fully exploit larger model capacity.

**SVG conventions learned:** The model consistently generates viewBox="0 0 24 24" (the dominant value in training data), xmlns namespace declarations, appropriate stroke/fill attributes, and valid path commands (M, L, C, Z). The 82.6% XML validity rate confirms strong SVG grammar internalization.

**Phase transitions:** We did not observe sudden capability jumps at specific scales. Improvement from Tiny to Medium is gradual. This may be because SVG is simpler than natural language, or because our model range (1M-91M) is too narrow to capture such transitions, which in NLP typically occur at billions of parameters.


## 7. Challenges Encountered

**1. Fixed learning rate failure:** The most significant challenge was discovering that LR=0.01 tuned on Tiny caused XL to achieve WORSE loss (1.37) than Tiny (1.23) under standard parameterization. The SP training curves showed clear instability: XL's loss spiked to 4.3 and oscillated for the first 2,500 steps. This initially appeared to suggest larger models hurt performance, which would have been an incorrect conclusion. The muP experiments proved this was entirely a hyperparameter transfer failure, as XL with muP achieved 0.97.

**2. XML validity in generation:** About 17% of generated SVGs had malformed XML, primarily from truncation at the max token limit. The model generates tokens without maintaining XML parse state, so it has no guarantee of producing valid XML. Our repair function improved validity to 82.6%, but grammar-constrained decoding would guarantee 100% validity.

**3. Context window limitations:** The 256-token context window is shorter than the median SVG (475 tokens). The model never sees complete SVGs during training for most examples. The star prefix completion failure (768 tokens, never terminating) is likely partly due to the model losing track of the polygon structure beyond its context window.

**4. Evaluation gap:** Our quantitative metrics capture syntax quality but not visual quality. T=1.0 had the highest render rate (83%) but produced abstract/noisy outputs, while T=0.8 had a lower render rate (33%) but produced the most visually coherent icons. Automated visual quality metrics like FID scores on rendered PNGs would better capture what matters.

**5. Compute constraints:** Training XL for 5 epochs took 6.3 hours, limiting our ability to run per-model LR sweeps, train for more epochs (the curve was still declining), test larger models, or perform systematic hyperparameter tuning.

**6. muP implementation complexity:** Applying muP correctly required changing attention scaling from 1/sqrt(d) to 1/d, using MuReadout (disabling weight tying), configuring base shapes with different width models, and using MuAdamW. The base shape configuration was particularly tricky with varying depths.

**7. Temperature bimodality:** At T=0.5, the model produced either 26-token (immediate EOS) or 793-token (max length) sequences with nothing in between. Low temperature concentrates probability on either EOS or the most common continuation pattern, making T=0.5 the least useful temperature.

**What we would do differently:**

1. Use a larger context window (512 or 1024 tokens)
2. Implement grammar-constrained decoding for guaranteed XML validity
3. Run per-model LR sweeps to establish an oracle baseline for muP comparison
4. Add FID-based visual quality metrics on rendered SVGs
5. Train for more epochs with learning rate restarts
6. Explore compute-optimal data/parameter allocation (Chinchilla-style)
7. Investigate conditional text-to-SVG generation using the svgen-500k dataset


## Report Structure Checklist

| Pages | Content | Figures |
|-------|---------|---------|
| 1 | Title, Introduction | |
| 2-3 | Part 1: Data | seq_length_histogram.png, example SVGs, stats table |
| 3-5 | Part 2: Scaling | lr_sweep.png, scaling_plot.png, training_curves.png |
| 5-7 | Part 3: muP | lr_sweep_sp_vs_mup.png, scaling_sp_vs_mup.png, mup_training_curves.png, scaling_extrapolation.png |
| 7-8 | Part 4: Generation | best_model_training_curve.png, grid_temperature_comparison.png, grid_prefix_completions.png, metrics table |
| 8-9 | Part 5: Analysis | Sections 1-7 from this notebook |
| 10 | References | Kaplan 2020, Hoffmann 2022, Yang 2022, Rodriguez 2023 |
